# PSET 4: State Mortality and Health Spending

This notebook implements all required data cleaning, merging, descriptive statistics, and regression steps.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path
from stargazer.stargazer import Stargazer

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


In [ ]:
# Paths (adjust DATA_DIR if your files are in another folder)
DATA_DIR = Path('.')
MORTALITY_FILE = DATA_DIR / 'mortality_data.csv'
INCOME_FILE = DATA_DIR / 'income_data.csv'
EDUC_DIR = DATA_DIR / 'education'
EXPND_DIR = DATA_DIR / 'expenditure'


## 1–7) Mortality data

In [ ]:
mort_data = pd.read_csv(MORTALITY_FILE)

# Keep only 1993-2015
mort_data = mort_data.loc[mort_data['year'].between(1993, 2015)].copy()

# Rename columns 4 through 11 (1-indexed) => positions 3..10 (0-indexed)
mort_vars = [
    'mort_rate', 'prob_death', 'ave_length_surv', 'num_of_surv',
    'num_of_deaths', 'num_years_lived', 'num_years_left', 'life_expec'
]
rename_map = dict(zip(mort_data.columns[3:11], mort_vars))
mort_data = mort_data.rename(columns=rename_map)

# Create Age2 from string before '-'
mort_data['Age2'] = pd.to_numeric(
    mort_data['Age'].astype(str).str.split('-').str[0],
    errors='coerce'
)

# Age groups: (0,18], (18,64], (64,max]
mort_data['age_group'] = pd.cut(
    mort_data['Age2'],
    bins=[0, 18, 64, mort_data['Age2'].max()],
    labels=['<=18', '19-64', '>64'],
    include_lowest=False,
    right=True
)

# Drop Age and Age2, reorder columns
mort_data = mort_data.drop(columns=['Age', 'Age2'])
mort_data = mort_data[['state', 'year', 'age_group'] + mort_vars]

# Aggregate mean by state-year-age_group (NaN excluded by default)
mort_data = (
    mort_data
    .groupby(['state', 'year', 'age_group'], as_index=False)[mort_vars]
    .mean()
)

# Fix state name issue and sort
mort_data['state'] = mort_data['state'].replace({'	Mississippi': 'Mississippi'})
mort_data = mort_data.sort_values(['state', 'year', 'age_group']).reset_index(drop=True)

mort_data.head()


## 8–9) Income data (wide to long)

In [ ]:
inc_data = pd.read_csv(INCOME_FILE)

# Identify year-varying columns that have a dot in the name, e.g. pinc.1993
dot_cols = [c for c in inc_data.columns if '.' in c]
stubnames = sorted({c.split('.')[0] for c in dot_cols})

# Wide to long
inc_data = (
    pd.wide_to_long(
        inc_data,
        stubnames=stubnames,
        i='state',
        j='year',
        sep='.',
        suffix='\d+'
    )
    .reset_index()
    .sort_values(['state', 'year'])
    .reset_index(drop=True)
)

inc_data.head()


## 10) Education data append

In [ ]:
educ_files = [EDUC_DIR / f'education_{yr}.csv' for yr in range(1993, 2007)]
educ_files.append(EDUC_DIR / 'education_0715.csv')

educ_data = pd.concat((pd.read_csv(f) for f in educ_files), ignore_index=True)

# Rename 3rd and 4th columns to phs and pcoll
educ_cols = list(educ_data.columns)
educ_data = educ_data.rename(columns={educ_cols[2]: 'phs', educ_cols[3]: 'pcoll'})

educ_data = educ_data.sort_values(['state', 'year']).reset_index(drop=True)
educ_data.head()


## 11) Expenditure data append

In [ ]:
expnd_files = [EXPND_DIR / f'expnd_{yr}.csv' for yr in range(1993, 2016)]

exp_frames = []
for f in expnd_files:
    df = pd.read_csv(f)

    # Normalize likely naming differences across years
    normalized = {c: c.strip().lower().replace(' ', '_') for c in df.columns}
    df = df.rename(columns=normalized)

    # Standardize common variants
    aliases = {
        'total_revenue': 'tot_revenue',
        'totrev': 'tot_revenue',
        'total_expenditure': 'tot_expnd',
        'total_expnd': 'tot_expnd',
        'totexpnd': 'tot_expnd',
        'publicwelfare': 'public_welfare',
        'public_welf': 'public_welfare'
    }
    df = df.rename(columns={c: aliases.get(c, c) for c in df.columns})
    exp_frames.append(df)

expnd_data = pd.concat(exp_frames, ignore_index=True)
expnd_data = expnd_data.sort_values(['state', 'year']).reset_index(drop=True)

# Make all expenditure variables numeric (except state/year)
for c in expnd_data.columns:
    if c not in ['state', 'year']:
        expnd_data[c] = pd.to_numeric(expnd_data[c], errors='coerce')

expnd_data.head()


## 12–15) Merge datasets

In [ ]:
# one-to-one merge
ndata = inc_data.merge(educ_data, on=['state', 'year'], how='inner', validate='one_to_one')

# one-to-one merge
ndata = ndata.merge(expnd_data, on=['state', 'year'], how='inner', validate='one_to_one')

# many-to-one merge (mortality has multiple age_group rows per state-year)
data = mort_data.merge(ndata, on=['state', 'year'], how='inner', validate='many_to_one')

# Remove intermediates
del mort_data, inc_data, educ_data, expnd_data, ndata

data.head()


## 16–18) Unit conversions and descriptive statistics

In [ ]:
money_cols = ['pinc', 'tot_revenue', 'taxes', 'tot_expnd', 'education', 'public_welfare', 'hospital', 'health']

for col in money_cols:
    if col in data.columns:
        data[col] = data[col] / 1e6

data['mort_rate'] = data['mort_rate'] * 1e4

desc_stats = data.describe(include='all').T
desc_stats


## 19–21) Regression specifications (age group 19-64)

In [ ]:
reg_data = data.loc[data['age_group'] == '19-64'].copy()

# Keep only rows where log terms are valid
reg_data = reg_data[(reg_data['health'] > 0) & (reg_data['hospital'] > 0) & (reg_data['pinc'] > 0)].copy()

spec_1 = smf.ols(
    'mort_rate ~ np.log(health) + np.log(hospital) + np.log(pinc) + phs + pcoll',
    data=reg_data
).fit()

spec_2 = smf.ols(
    'mort_rate ~ np.log(health) + np.log(hospital) + np.log(pinc) + phs + pcoll + C(state)',
    data=reg_data
).fit()

spec_3 = smf.ols(
    'mort_rate ~ np.log(health) + np.log(hospital) + np.log(pinc) + phs + pcoll + C(state) + C(year)',
    data=reg_data
).fit()

print(spec_1.summary())


## 22) Stargazer regression table

In [ ]:
stargazer_table = Stargazer([spec_1, spec_2, spec_3])
stargazer_table.title('Mortality Rate Regressions (Age Group: 19-64)')
stargazer_table.custom_columns(['Spec 1', 'Spec 2', 'Spec 3'], [1, 1, 1])
stargazer_table
